In [1]:
import os
import sys
import pyspark
from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel
# 1. CẤU HÌNH MÔI TRƯỜNG
MY_JAVA_HOME = r"D:\java\openjdk-17.0.18b8"
MY_HADOOP_HOME = r"D:\java\hadoop-3.4.3"
os.environ["JAVA_HOME"] = MY_JAVA_HOME
os.environ["HADOOP_HOME"] = MY_HADOOP_HOME
os.environ["SPARK_HOME"] = os.path.dirname(pyspark.__file__)
sys.path.append(os.path.join(MY_HADOOP_HOME, "bin"))
# 2. KHỞI TẠO SPARK SESSION
spark = SparkSession.builder \
    .appName('MetroPT3_SQL_Analysis_Group16') \
    .master("local[*]") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://10.125.222.18:9000") \
    .config('spark.sql.shuffle.partitions', '8') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .getOrCreate()
print("-> Trạng thái: Spark Session cho SQL đã sẵn sàng!")
# 3. ĐỌC DỮ LIỆU SẠCH VÀ TẠO ĐẶC TRƯNG THỜI GIAN
HDFS_CLEAN_FOR_SQL = "hdfs://10.125.222.18:9000/user/bigdata/cleaned/metropt3_clean_for_sql"
print("-> Đang nạp dữ liệu Parquet từ HDFS...")
df = spark.read.parquet(HDFS_CLEAN_FOR_SQL)
# Cột 'timestamp' đã là dạng thời gian chuẩn, chỉ cần trích xuất thẳng ra giờ, tháng, ngày
df = df.withColumn('hour',    F.hour('timestamp')) \
       .withColumn('month',   F.month('timestamp')) \
       .withColumn('weekday', F.dayofweek('timestamp')) \
       .withColumn('date',    F.to_date('timestamp'))
# 4. LƯU TRỮ VÀO BỘ NHỚ ĐỆM (CACHE) & TẠO VIEW TRUY VẤN
print("-> Đang đẩy dữ liệu vào RAM (Persist) để tăng tốc 20 câu query...")
df.persist(StorageLevel.MEMORY_AND_DISK)
# Kích hoạt hành động (Trigger) để ép Spark thực sự nạp dữ liệu vào RAM
total_rows = df.count()
print(f"-> Cached thành công: {total_rows:,} rows")
# Tạo bảng ảo tên là 'sensor' để sử dụng cú pháp spark.sql()
df.createOrReplaceTempView('sensor')
print("=== SETUP HOÀN TẤT. SẴN SÀNG CHẠY CÁC CÂU QUERY ===")

-> Trạng thái: Spark Session cho SQL đã sẵn sàng!
-> Đang nạp dữ liệu Parquet từ HDFS...
-> Đang đẩy dữ liệu vào RAM (Persist) để tăng tốc 20 câu query...
-> Cached thành công: 1,516,948 rows
=== SETUP HOÀN TẤT. SẴN SÀNG CHẠY CÁC CÂU QUERY ===


In [2]:
# Q9: PHÂN TÍCH CÁC PHIÊN VẬN HÀNH LIÊN TỤC DÀI NHẤT
q9 = spark.sql('''
WITH session_edges AS (
    SELECT
        timestamp,
        COMP,
        Motor_current,
        LAG(COMP,1,1) OVER(ORDER BY timestamp) AS prev_comp FROM sensor),
session_blocks AS (
    SELECT
        timestamp,
        COMP,
        Motor_current,
        SUM(
            CASE
                WHEN COMP = 0 AND prev_comp = 1 THEN 1
                ELSE 0 END
        ) OVER(ORDER BY timestamp) AS session_id
    FROM session_edges)
SELECT
    session_id,
    MIN(timestamp) AS session_start,
    MAX(timestamp) AS session_end,
    COUNT(*) AS duration_seconds,
    ROUND(AVG(Motor_current),2) AS avg_current_in_session
FROM session_blocks
WHERE COMP = 0
GROUP BY session_id
ORDER BY duration_seconds DESC
LIMIT 10''')
print("\n========== EXECUTION PLAN ==========")
q9.explain(True)
print("\n--- TOP 10 PHIÊN VẬN HÀNH LIÊN TỤC DÀI NHẤT ---")
df_q9 = q9.toPandas()
try:
    display(df_q9)
except NameError:
    print(df_q9.to_string(index=False))


========== EXECUTION PLAN ==========
== Parsed Logical Plan ==
CTE [session_edges, session_blocks]
:  :- 'SubqueryAlias session_edges
:  :  +- 'Project ['timestamp, 'COMP, 'Motor_current, 'LAG('COMP, 1, 1) windowspecdefinition('timestamp ASC NULLS FIRST, unspecifiedframe$()) AS prev_comp#715]
:  :     +- 'UnresolvedRelation [sensor], [], false
:  +- 'SubqueryAlias session_blocks
:     +- 'Project ['timestamp, 'COMP, 'Motor_current, 'SUM(CASE WHEN (('COMP = 0) AND ('prev_comp = 1)) THEN 1 ELSE 0 END) windowspecdefinition('timestamp ASC NULLS FIRST, unspecifiedframe$()) AS session_id#716]
:        +- 'UnresolvedRelation [session_edges], [], false
+- 'GlobalLimit 10
   +- 'LocalLimit 10
      +- 'Sort ['duration_seconds DESC NULLS LAST], true
         +- 'Aggregate ['session_id], ['session_id, 'MIN('timestamp) AS session_start#711, 'MAX('timestamp) AS session_end#712, 'COUNT(1) AS duration_seconds#713, 'ROUND('AVG('Motor_current), 2) AS avg_current_in_session#714]
            +- 'Filter 

,session_id,session_start,session_end,duration_seconds,avg_current_in_session
0,11818,2020-06-22 15:05:41,2020-06-25 05:08:35,18518,5.57
1,10042,2020-06-05 09:48:30,2020-06-08 13:54:18,17951,5.49
2,4490,2020-04-18 00:23:59,2020-04-19 01:55:36,9272,5.67
3,2746,2020-03-28 07:22:24,2020-03-30 07:41:06,8105,5.63
4,6807,2020-05-19 22:22:17,2020-05-20 23:02:33,7830,5.48
5,3481,2020-04-12 11:50:31,2020-04-12 23:36:00,4271,5.80
6,1925,2020-03-12 00:16:06,2020-03-12 11:49:50,4199,5.83
7,6459,2020-05-13 13:44:04,2020-05-14 04:36:21,4139,5.32
8,9678,2020-05-29 23:14:56,2020-05-30 05:56:46,2433,5.58
9,2683,2020-03-27 07:12:10,2020-03-27 11:38:11,1611,5.77
